# SC-GRC Hybrid — Notebook 04 (FIXED)

This notebook completes the final signal shard (questions **253–333**) only when those rows are missing, then trains the SC-GRC escalation router after all **333** signal rows exist.

Fixes included:
- KB-only retrieval; no scraped-data directory.
- Reranker scores are flattened to 1-D scalars.
- K=3 samples are generated **sequentially** to avoid the T4 CUDA OOM caused by batched `num_return_sequences=3`.
- Existing completed signal rows are never recomputed.
- The original BERTScore floor of 0.85 is used when it produces two classes. If it produces only one class, the notebook chooses the highest observed threshold below 0.85 that leaves at least 10% of the 333 items in each class and records that threshold.
- If all 333 rows already exist, the notebook skips SLM/RAG generation entirely and only builds the router.


In [1]:
%%capture
import os, subprocess, sys
cuda_lib = "/usr/local/cuda/lib64"
ld = os.environ.get("LD_LIBRARY_PATH", "")
if cuda_lib not in ld:
    os.environ["LD_LIBRARY_PATH"] = f"{cuda_lib}:{ld}"
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-cache-dir",
    "torch==2.3.1","torchvision==0.18.1","--extra-index-url","https://download.pytorch.org/whl/cu121"])
# Pin Triton to the version torch==2.3.1 was built against. Kaggle base images often
# ship a newer Triton (3.x) that dropped the triton.ops submodule, which makes
# bitsandbytes' triton_based_modules.py crash on import with:
#   "No module named 'triton.ops'"
# Force-reinstalling the matching Triton here fixes that before bitsandbytes/transformers load.
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-cache-dir","--force-reinstall",
    "triton==2.3.1"])
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-cache-dir",
    "pandas==2.2.2","scipy==1.13.1","bitsandbytes==0.44.1",
    "transformers==4.46.3","peft==0.13.2","accelerate==0.34.2","sentencepiece","sacrebleu",
    "rapidfuzz","openpyxl","langchain-text-splitters","faiss-gpu-cu12","FlagEmbedding",
    "sentence-transformers","rank_bm25","scikit-learn","bert-score","tqdm"])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 176.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 232.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 74.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 151.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 198.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 204.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 213.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 227.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 255.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 244.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 194.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.3.1+cu121 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 MB 183.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 381.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.3.1+cu121 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 191.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 296.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 268.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 169.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 205.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 363.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 359.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 366.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 204.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 142.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 327.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.13.1 which is incompatible.


In [2]:
# ========================= USER CONFIG =========================
# EDIT THIS: mount the Kaggle Dataset created from Notebook 03 /kaggle/working/sc_grc_state.
INPUT_STATE_DIR = "/kaggle/input/datasets/bxgdhdgsg/hbr333/sc_grc_state"

# Only needed if eval_333.csv is missing from the state dataset.
GROUND_TRUTH_XLSX = "/kaggle/input/datasets/mohuaakter/ewu-dataset-jsonl-pairs/University_Chatbot_Questions_1110_GROUND_TRUTHS_VERIFIED.xlsx"

# Used only if Notebook 04 discovers missing rows 253–333.
MODEL_ID_SLM = "Qwen/Qwen2.5-3B-Instruct"
SLM_ADAPTER_DIR = "/kaggle/input/datasets/mrnotalent/ewu-qwen-adapter-checkpoint1632"
KB_DIR = "/kaggle/input/datasets/mohuaakter/ewu-dataset-jsonl-pairs/KB/KB"  # KB ONLY
WORK_STATE_DIR = "/kaggle/working/sc_grc_state"

TAIL_START = 252  # zero-based -> question 253
TAIL_END = 333    # exclusive -> question 333
SEED = 42
TOP_K_DENSE = 10
TOP_K_SPARSE = 10
TOP_K_FINAL = 4
K_SAMPLES = 3
SC_GRC_TEMPERATURE = 0.7
SC_GRC_TOP_P = 0.9
MAX_NEW_TOKENS = 128
ORIGINAL_BERTSCORE_FLOOR = 0.85
MIN_CLASS_FRACTION = 0.10

print("Config loaded.")


Config loaded.


In [3]:
import os, json, re, random, shutil, time, gc
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

os.makedirs(WORK_STATE_DIR, exist_ok=True)
if os.path.isdir(INPUT_STATE_DIR) and os.path.abspath(INPUT_STATE_DIR) != os.path.abspath(WORK_STATE_DIR):
    for name in os.listdir(INPUT_STATE_DIR):
        src=os.path.join(INPUT_STATE_DIR,name); dst=os.path.join(WORK_STATE_DIR,name)
        if os.path.isfile(src) and not os.path.exists(dst): shutil.copy2(src,dst)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("DEVICE:", "cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

SIGNAL_FILE=os.path.join(WORK_STATE_DIR,"sc_grc_signals.jsonl")
if not os.path.exists(SIGNAL_FILE): raise FileNotFoundError(f"Cannot find {SIGNAL_FILE}. Mount Notebook 03 state first.")
rows={}
with open(SIGNAL_FILE,encoding="utf-8") as f:
    for line in f:
        if line.strip():
            r=json.loads(line); rows[int(r["idx"])]=r
print(f"Existing signal rows: {len(rows)} / 333")
print("Final-shard rows present:", sum(i in rows for i in range(TAIL_START,TAIL_END)), "/ 81")
missing_early=[i for i in range(0,TAIL_START) if i not in rows]
if missing_early: raise RuntimeError("Notebook 04 expects rows 0..251 already present. Missing early rows: "+str(missing_early[:20]))


DEVICE: cuda
GPU: Tesla T4
Existing signal rows: 252 / 333
Final-shard rows present: 0 / 81


In [4]:
# ========================= LOAD 333-EVAL METADATA =========================
EVAL_CSV=os.path.join(WORK_STATE_DIR,"eval_333.csv")
if os.path.exists(EVAL_CSV):
    eval_df=pd.read_csv(EVAL_CSV)
else:
    print("eval_333.csv not found; reconstructing from the ground-truth workbook.")
    LANGUAGE_COLUMNS={"english":"English Query","bangla":"Bangla Query","banglish":"Banglish Query"}
    DIFFICULTY_ALIAS={"Easy":"Simple","Medium":"Normal","Hard":"Complex"}
    corpus=pd.read_excel(GROUND_TRUTH_XLSX,sheet_name="Question Corpus")
    expanded=[]
    for _,r in corpus.iterrows():
        for language,qcol in LANGUAGE_COLUMNS.items():
            q=r[qcol]
            if pd.isna(q) or not str(q).strip(): continue
            expanded.append({"Q#":r["Q#"],"language":language,"query":str(q).strip(),"Category":r.get("Category",""),"Difficulty":r.get("Difficulty",""),"Difficulty Group":DIFFICULTY_ALIAS.get(str(r.get("Difficulty","")),str(r.get("Difficulty",""))),"ground_truth":str(r["Ground Truth Answer (Canonical English)"]).strip()})
    expanded_df=pd.DataFrame(expanded)
    chosen_qnums=(corpus.groupby(["Category","Difficulty"],dropna=False)["Q#"].apply(lambda s:s.sample(n=min(1,len(s)),random_state=SEED)).reset_index(drop=True).tolist())
    eval_df=expanded_df[expanded_df["Q#"].isin(chosen_qnums)].sample(frac=1,random_state=SEED).reset_index(drop=True)
    if len(eval_df)!=333: raise ValueError(f"Expected 333 eval rows, got {len(eval_df)}")
    eval_df.to_csv(EVAL_CSV,index=False,encoding="utf-8-sig")
if len(eval_df)!=333: raise ValueError(f"eval_333.csv has {len(eval_df)} rows, expected 333.")
test_prompts_eval=eval_df["query"].astype(str).tolist()
test_references_eval=eval_df["ground_truth"].astype(str).tolist()
test_langs_eval=eval_df["language"].astype(str).tolist()
print("Eval set verified:",len(test_prompts_eval))


Eval set verified: 333


In [5]:
# ========================= OPTIONAL: COMPLETE ONLY MISSING 253–333 SIGNALS =========================
pending_tail=[i for i in range(TAIL_START,TAIL_END) if i not in rows]
print("Missing final-shard rows:",len(pending_tail))

if pending_tail:
    print("Loading SLM/RAG stack because some Notebook-04 rows are missing...")
    from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig,AutoModelForSequenceClassification
    from peft import PeftModel
    import faiss
    from rank_bm25 import BM25Okapi
    from sentence_transformers import SentenceTransformer
    from FlagEmbedding import FlagReranker
    DEVICE="cuda" if torch.cuda.is_available() else "cpu"
    KB_JSON=os.path.join(WORK_STATE_DIR,"kb.json"); KB_META=os.path.join(WORK_STATE_DIR,"kb_meta.json"); FAISS_PATH=os.path.join(WORK_STATE_DIR,"dense_index.faiss")
    if os.path.exists(KB_JSON) and os.path.exists(KB_META) and os.path.exists(FAISS_PATH):
        kb_chunks=json.load(open(KB_JSON,encoding="utf-8")); meta=json.load(open(KB_META,encoding="utf-8")); kb_sources=meta["sources"]; dense_index=faiss.read_index(FAISS_PATH)
    else:
        from langchain_text_splitters import MarkdownTextSplitter
        SKIP_KEYS={"navigation_links","page_info","summary"}
        def _flatten_record(d):
            return ". ".join(f"{k.replace("_"," ")}: {v}" for k,v in d.items() if not isinstance(v,(dict,list)) and v not in (None,""))
        def _walk_json(obj,file_tag,context=""):
            docs=[]
            if isinstance(obj,dict):
                if obj and all(not isinstance(v,(dict,list)) for v in obj.values()):
                    t=_flatten_record(obj)
                    if t: docs.append((file_tag,f"{context} {t}".strip()))
                    return docs
                for k,v in obj.items():
                    if k in SKIP_KEYS: continue
                    docs.extend(_walk_json(v,file_tag,f"{context} {k}".strip()))
            elif isinstance(obj,list):
                for item in obj: docs.extend(_walk_json(item,file_tag,context))
            return docs
        if not os.path.isdir(KB_DIR): raise FileNotFoundError(f"KB_DIR not found: {KB_DIR}")
        splitter=MarkdownTextSplitter(chunk_size=500,chunk_overlap=80); kb_chunks=[]; kb_sources=[]
        for root,_,files in os.walk(KB_DIR):
            for fn in sorted(files):
                p=os.path.join(root,fn); rel=os.path.relpath(p,KB_DIR)
                try:
                    if fn.lower().endswith((".md",".markdown",".txt")):
                        txt=open(p,encoding="utf-8",errors="ignore").read().strip()
                        for ch in splitter.split_text(txt):
                            if ch.strip(): kb_chunks.append(ch); kb_sources.append(rel)
                    elif fn.lower().endswith(".json"):
                        obj=json.load(open(p,encoding="utf-8"))
                        for src,txt in _walk_json(obj,rel):
                            if txt.strip(): kb_chunks.append(txt); kb_sources.append(src)
                except Exception as e: print("[skip]",p,type(e).__name__,e)
        if not kb_chunks: raise RuntimeError("No KB chunks found.")
        tmp_embed=SentenceTransformer("BAAI/bge-m3",device=DEVICE); tmp_embed.max_seq_length=512
        embs=np.asarray(tmp_embed.encode(kb_chunks,batch_size=16,normalize_embeddings=False,show_progress_bar=True),dtype="float32"); faiss.normalize_L2(embs)
        dense_index=faiss.IndexFlatIP(embs.shape[1]); dense_index.add(embs)
        json.dump(kb_chunks,open(KB_JSON,"w",encoding="utf-8"),ensure_ascii=False)
        json.dump({"sources":kb_sources,"n_chunks":len(kb_chunks)},open(KB_META,"w",encoding="utf-8"),ensure_ascii=False,indent=2)
        faiss.write_index(dense_index,FAISS_PATH); del tmp_embed,embs; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    embedder=SentenceTransformer("BAAI/bge-m3",device=DEVICE); embedder.max_seq_length=512
    reranker=FlagReranker("BAAI/bge-reranker-v2-m3",use_fp16=True,device=DEVICE)
    bm25=BM25Okapi([re.findall(r"[\w\u0980-\u09FF]+",c.lower()) for c in kb_chunks])
    def embed_texts(texts,batch_size=16): return np.asarray(embedder.encode(texts,batch_size=batch_size,normalize_embeddings=False,show_progress_bar=False),dtype="float32")
    def hybrid_retrieve(query, top_k=TOP_K_FINAL):
        q = embed_texts([query])
        faiss.normalize_L2(q)

        _, di = dense_index.search(
            q,
            min(TOP_K_DENSE, len(kb_chunks))
        )

        ss = bm25.get_scores(
            re.findall(r"[\w\u0980-\u09FF]+", query.lower())
        )
        si = np.argsort(ss)[::-1][:min(TOP_K_SPARSE, len(kb_chunks))]

        ids = sorted(set(di[0].tolist()) | set(si.tolist()))
        pairs = [[query, kb_chunks[i]] for i in ids]

        raw_scores = reranker.compute_score(pairs, normalize=True)

        # Make the reranker output a 1-D scalar score array.
        rs = np.asarray(raw_scores, dtype=np.float32).reshape(-1)

        if len(rs) != len(ids):
            raise RuntimeError(
                f"Reranker returned {len(rs)} scores for {len(ids)} candidate chunks. "
                f"Raw shape={np.asarray(raw_scores).shape}"
            )

        order = np.argsort(rs)[::-1][:top_k]

        return [
            {
                'source': kb_sources[ids[int(j)]],
                'chunk': kb_chunks[ids[int(j)]],
                'score': float(rs[int(j)])
            }
            for j in order
        ]
    SYSTEM_PROMPT=("You are the East West University (EWU) student support assistant. Answer the student's question directly, accurately, and helpfully. Do not invent university policies or facts. If the provided context is insufficient, say so clearly.")
    def build_rag_prompt(query,retrieved):
        ctx="\n\n".join(f"[{r['source']}]\n{r['chunk']}" for r in retrieved) if retrieved else "(no relevant context retrieved)"
        user=f"Context:\n{ctx}\n\nQuestion: {query}\n\nAnswer using only the context above, following the system rules."
        return f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n"
    bnb_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_use_double_quant=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16)
    base_slm=AutoModelForCausalLM.from_pretrained(MODEL_ID_SLM,quantization_config=bnb_config,device_map="auto",trust_remote_code=True,torch_dtype=torch.float16,attn_implementation="eager")
    slm=PeftModel.from_pretrained(base_slm,SLM_ADAPTER_DIR).eval(); slm.config.use_cache=True
    slm_tokenizer=AutoTokenizer.from_pretrained(SLM_ADAPTER_DIR,trust_remote_code=True,padding_side="right")
    if slm_tokenizer.pad_token is None: slm_tokenizer.pad_token=slm_tokenizer.eos_token
    @torch.no_grad()
    def generate_k_samples(query, retrieved, k=K_SAMPLES):
        p = build_rag_prompt(query, retrieved)

        # Prevent very long RAG contexts from causing another large VRAM spike.
        enc = slm_tokenizer(
            p,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        ).to(slm.device)

        samples = []
        n = enc["input_ids"].shape[1]

        for _ in range(k):
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            out = slm.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=SC_GRC_TEMPERATURE,
                top_p=SC_GRC_TOP_P,
                num_return_sequences=1,
                pad_token_id=slm_tokenizer.pad_token_id,
                use_cache=True,
            )

            samples.append(
                slm_tokenizer.decode(
                    out[0][n:],
                    skip_special_tokens=True
                ).strip()
            )

            del out
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return samples
    def pick_representative_sample(samples):
        if len(samples)<=1: return samples[0] if samples else ""
        e=embed_texts(samples); faiss.normalize_L2(e); c=e.mean(axis=0,keepdims=True); faiss.normalize_L2(c); return samples[int(np.argmax((e@c.T).reshape(-1)))]
    def self_consistency_score(samples):
        if len(samples)<2: return 1.0
        e=embed_texts(samples); faiss.normalize_L2(e); sims=[float(np.dot(e[i],e[j])) for i in range(len(samples)) for j in range(i+1,len(samples))]; return float(np.mean(sims)) if sims else 1.0
    NLI_MODEL_ID="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
    nli_tok=AutoTokenizer.from_pretrained(NLI_MODEL_ID); nli=AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_ID).to("cpu").eval()
    NLI_ENT=int([k for k,v in nli.config.id2label.items() if v.lower().startswith("entail")][0])
    def _split_sentences(t): return [x.strip() for x in re.split(r"(?<=[।.!?])\s+",t.strip()) if x.strip()]
    @torch.no_grad()
    def groundedness_score(answer,retrieved):
        sents=_split_sentences(answer)
        if not sents or not retrieved: return 0.0
        ctx=" ".join(r["chunk"] for r in retrieved); vals=[]
        for s in sents:
            enc=nli_tok(ctx,s,truncation=True,max_length=512,return_tensors="pt"); pred=int(nli(**enc).logits.argmax(dim=-1).item()); vals.append(pred==NLI_ENT)
        return float(np.mean(vals))
    def compute_cmi(text):
        ban=re.compile(r"[\u0980-\u09FF]"); lat=re.compile(r"[A-Za-z]+"); words=re.findall(r"[\w\u0980-\u09FF]+",text)
        if not words: return 0.0
        nb=sum(1 for w in words if ban.search(w)); nl=sum(1 for w in words if lat.fullmatch(w)); n=nb+nl
        return 0.0 if n==0 else 100.0*(n-max(nb,nl))/n
    with open(SIGNAL_FILE,"a",encoding="utf-8") as f:
        for idx in tqdm(pending_tail,desc="SC-GRC signals 253–333"):
            q=test_prompts_eval[idx]; retrieved=hybrid_retrieve(q); samples=generate_k_samples(q,retrieved); rep=pick_representative_sample(samples); sc=self_consistency_score(samples); gs=groundedness_score(rep,retrieved); cmi=compute_cmi(q)
            rows[idx]={"idx":idx,"query":q,"lang":test_langs_eval[idx],"representative_answer":rep,"samples":samples,"self_consistency":sc,"groundedness":gs,"cmi":cmi}
            f.write(json.dumps(rows[idx],ensure_ascii=False)+"\n"); f.flush()
    print("Signal shard completion:",len(rows),"/ 333")
    del slm,base_slm,embedder,reranker,nli
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print("All 333 signal rows already exist. Skipping SLM/RAG generation.")


Missing final-shard rows: 81
Loading SLM/RAG stack because some Notebook-04 rows are missing...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

SC-GRC signals 253–333:   0%|          | 0/81 [00:00<?, ?it/s]


initial target device: 100%|██████████| 2/2 [00:18<00:00,  9.43s/it]

Chunks:   0%|          | 0/2 [00:00<?, ?it/s]You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.

Chunks:  50%|█████     | 1/2 [00:02<00:02,  2.33s/it]You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.

Chunks: 100%|██████████| 2/2 [00:02<00:00,  1.36s/it]

Chunks: 100%|██████████| 2/2 [00:00<00:00,  4.55it/s]

Chunks: 100%|██████████| 2/2 [00:00<00:00,  4.40it/s]

Chunks: 100%|██████████| 2/2 [00:00<00:00,  5.15it/s]

Chunks: 100%|██████████| 2/2 [00:00<00:00,  4.82it/s]

Chunks: 100%|██████████| 2/2 [00:00<00:00,  4.51it/s]

Chunks: 100%|██████████| 2/2 [00:00

Signal shard completion: 333 / 333


## Train the escalation router after all 333 signals exist

The earlier router used a fixed BERTScore floor of 0.85. In the failed run, all items fell below that floor, so logistic regression saw only class `1` and stopped. This cell uses 0.85 when it is valid; otherwise it chooses the closest supported threshold below 0.85 with at least 10% in each class and writes the exact threshold into the state.


In [6]:
if len(rows)!=333: raise RuntimeError(f"Cannot train router yet: only {len(rows)} / 333 signal rows are present.")
from bert_score import score as bert_score_fn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import joblib
ordered=pd.DataFrame([rows[i] for i in sorted(rows)]).sort_values("idx").reset_index(drop=True)
if "rep_bertscore_f1" in ordered.columns and ordered["rep_bertscore_f1"].notna().all():
    print("Reusing existing rep_bertscore_f1 values.")
else:
    print("Computing multilingual BERTScore F1 on 333 representative answers (CPU).")
    _,_,bert_f1=bert_score_fn(ordered["representative_answer"].astype(str).tolist(),test_references_eval,model_type="bert-base-multilingual-cased",device="cpu",batch_size=4,verbose=False)
    ordered["rep_bertscore_f1"]=bert_f1.tolist()
scores=ordered["rep_bertscore_f1"].astype(float).to_numpy()
print("\nBERTScore distribution:")
print(pd.Series(scores).describe())
print(f"min={scores.min():.4f} mean={scores.mean():.4f} max={scores.max():.4f}")
labels_original=(scores<ORIGINAL_BERTSCORE_FLOOR).astype(int); counts_original=np.bincount(labels_original,minlength=2)
print("Original 0.85 label counts [0,1]:",counts_original.tolist())
chosen_threshold=ORIGINAL_BERTSCORE_FLOOR; threshold_source="original_fixed_0.85"
if np.unique(labels_original).size<2:
    min_class_count=max(5,int(np.ceil(MIN_CLASS_FRACTION*len(scores))))
    candidates=sorted(set(float(x) for x in np.round(scores,4) if float(x)<ORIGINAL_BERTSCORE_FLOOR),reverse=True)
    chosen_threshold=None
    for t in candidates:
        lab=(scores<t).astype(int); cc=np.bincount(lab,minlength=2)
        if cc[0]>=min_class_count and cc[1]>=min_class_count:
            chosen_threshold=float(t); break
    if chosen_threshold is None: raise RuntimeError("Could not find a supported threshold below 0.85 with two classes.")
    threshold_source="data_supported_fallback_below_0.85"
ordered["needs_escalation"]=(ordered["rep_bertscore_f1"].astype(float)<chosen_threshold).astype(int)
y=ordered["needs_escalation"].astype(int)
print(f"\nChosen threshold: {chosen_threshold:.4f} ({threshold_source})")
print("Final label counts:",y.value_counts().sort_index().to_dict())
if y.nunique()<2: raise RuntimeError("Router target still has one class; refusing to save a fake router.")
X=ordered[["self_consistency","groundedness","cmi"]].copy(); X["inv_consistency"]=1.0-X["self_consistency"]; X["inv_groundedness"]=1.0-X["groundedness"]; X=X[["inv_consistency","inv_groundedness","cmi"]]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.40,random_state=SEED,stratify=y)
clf=LogisticRegression(class_weight="balanced",random_state=SEED,max_iter=1000).fit(Xtr,ytr)
ordered["escalate_pred"]=clf.predict(X); ordered["escalate_proba"]=clf.predict_proba(X)[:,1]
router_csv=os.path.join(WORK_STATE_DIR,"sc_grc_signals_with_router.csv"); router_joblib=os.path.join(WORK_STATE_DIR,"sc_grc_router.joblib"); router_cfg=os.path.join(WORK_STATE_DIR,"sc_grc_router_config.json"); diagnostics_csv=os.path.join(WORK_STATE_DIR,"sc_grc_router_diagnostics.csv")
ordered.to_csv(router_csv,index=False,encoding="utf-8-sig"); joblib.dump(clf,router_joblib)
diagnostics={"n_samples":int(len(ordered)),"original_bertscore_floor":float(ORIGINAL_BERTSCORE_FLOOR),"chosen_bertscore_threshold":float(chosen_threshold),"threshold_source":threshold_source,"min_class_fraction":float(MIN_CLASS_FRACTION),"class_0_count":int((y==0).sum()),"class_1_count":int((y==1).sum()),"train_accuracy":float(clf.score(Xtr,ytr)),"heldout_accuracy":float(clf.score(Xte,yte)),"k_samples":int(K_SAMPLES),"temperature":float(SC_GRC_TEMPERATURE),"top_p":float(SC_GRC_TOP_P),"top_k_dense":int(TOP_K_DENSE),"top_k_sparse":int(TOP_K_SPARSE),"top_k_final":int(TOP_K_FINAL)}
json.dump(diagnostics,open(router_cfg,"w",encoding="utf-8"),indent=2)
pd.DataFrame([diagnostics]).to_csv(diagnostics_csv,index=False)
print("\n================ ROUTER READY ================")
print("rows:",len(ordered)); print("escalation labels:",int(y.sum()),"of",len(y)); print("escalation rate:",round(float(y.mean())*100,2),"%"); print("held-out router accuracy:",round(float(clf.score(Xte,yte)),3))
print("\nSaved:"); print(router_csv); print(router_joblib); print(router_cfg); print(diagnostics_csv)


Computing multilingual BERTScore F1 on 333 representative answers (CPU).


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]


BERTScore distribution:
count    333.000000
mean       0.598983
std        0.042607
min        0.516390
25%        0.570688
50%        0.594290
75%        0.616341
max        0.727208
dtype: float64
min=0.5164 mean=0.5990 max=0.7272
Original 0.85 label counts [0,1]: [0, 333]

Chosen threshold: 0.6627 (data_supported_fallback_below_0.85)
Final label counts: {0: 34, 1: 299}

================ ROUTER READY ================
rows: 333
escalation labels: 299 of 333
escalation rate: 89.79 %
held-out router accuracy: 0.582

Saved:
/kaggle/working/sc_grc_state/sc_grc_signals_with_router.csv
/kaggle/working/sc_grc_state/sc_grc_router.joblib
/kaggle/working/sc_grc_state/sc_grc_router_config.json
/kaggle/working/sc_grc_state/sc_grc_router_diagnostics.csv


In [7]:
# ========================= FINAL SANITY CHECK =========================
required=["sc_grc_signals.jsonl","eval_333.csv","sc_grc_signals_with_router.csv","sc_grc_router.joblib","sc_grc_router_config.json"]
print("\nOutput files:")
for name in required:
    path=os.path.join(WORK_STATE_DIR,name); print(("OK   " if os.path.exists(path) else "MISS "),name)
if all(os.path.exists(os.path.join(WORK_STATE_DIR,x)) for x in required):
    print("\nNotebook 04 completed successfully.")
    print("Save /kaggle/working/sc_grc_state as a Kaggle Dataset and use it for Notebook 05 INPUT_STATE_DIR.")
else:
    print("\nNotebook 04 is not complete; inspect missing outputs above.")



Output files:
OK    sc_grc_signals.jsonl
OK    eval_333.csv
OK    sc_grc_signals_with_router.csv
OK    sc_grc_router.joblib
OK    sc_grc_router_config.json

Notebook 04 completed successfully.
Save /kaggle/working/sc_grc_state as a Kaggle Dataset and use it for Notebook 05 INPUT_STATE_DIR.
